In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px


In [3]:
URL = 'https://raw.githubusercontent.com/hovhannisyan91/data_analytics_with_python/refs/heads/main/data/rfm/orders.csv'
df = pd.read_csv(URL, parse_dates=['order_date'])
df.head()

,customer_id,order_date,revenue
0,Mr. Brion Stark Sr.,2004-12-20,32
1,Ethyl Botsford,2005-05-02,36
2,Hosteen Jacobi,2004-03-06,116
3,Mr. Edw Frami,2006-03-15,99
4,Josef Lemke,2006-08-14,76


In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)
px.defaults.template = "plotly_white"

## Revenue Distribution

In [5]:
fig = px.histogram(
    df,
    x='revenue',
    nbins=30,
    title='Histogram of Revenue'
)

fig.update_layout(
    xaxis_title='Revenue',
    yaxis_title='Count'
)

fig.show()

In [6]:
df.head()

,customer_id,order_date,revenue
0,Mr. Brion Stark Sr.,2004-12-20,32
1,Ethyl Botsford,2005-05-02,36
2,Hosteen Jacobi,2004-03-06,116
3,Mr. Edw Frami,2006-03-15,99
4,Josef Lemke,2006-08-14,76


```sql
SELECT 
    customer_id,
    SUM(revenue) as revenue,
    count(customer_id) as frequency,
    (SELECT MAX(order_date) FROM df) - MAX(order_date) as recency
FROM df
GROUP BY customer_id
```

In [7]:
max_date = df['order_date'].max()
max_date

Timestamp('2006-12-30 00:00:00')

In [8]:
max_date = df['order_date'].max()

rfm = df.groupby('customer_id').agg(
    {
        'order_date': lambda date: (max_date - date.max()).days,
        'customer_id': lambda num: len(num),
        'revenue': lambda price: price.sum()
    }
)

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.reset_index(inplace=True)
rfm.head()

,customer_id,Recency,Frequency,Monetary
0,Abbey O'Reilly DVM,204,6,472
1,Add Senger,139,3,340
2,Aden Lesch Sr.,193,4,405
3,Admiral Senger,131,5,448
4,Agness O'Keefe,89,9,843


In [9]:
rfm['R'] = pd.qcut(rfm['Recency'], 4, ['4', '3', '2', '1'])

rfm['F'] = pd.qcut(rfm['Frequency'], 4, ['1', '2', '3', '4'])

rfm['M'] = pd.qcut(rfm['Monetary'], 4, ['1', '2', '3', '4'])

In [10]:
rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M
0,Abbey O'Reilly DVM,204,6,472,3,3,3
1,Add Senger,139,3,340,3,1,2
2,Aden Lesch Sr.,193,4,405,3,2,2
3,Admiral Senger,131,5,448,4,2,3
4,Agness O'Keefe,89,9,843,4,4,4


In [ ]:
weights = {
    "R": 0.7,
    "F": 0.2,
    "M": 0.1
}

weights["R"]

0.7

In [11]:
rfm['total_score'] = rfm['R'].astype(int) + rfm['F'].astype(int) + rfm['M'].astype(int)
rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M,total_score
0,Abbey O'Reilly DVM,204,6,472,3,3,3,9
1,Add Senger,139,3,340,3,1,2,6
2,Aden Lesch Sr.,193,4,405,3,2,2,7
3,Admiral Senger,131,5,448,4,2,3,9
4,Agness O'Keefe,89,9,843,4,4,4,12


In [15]:
rfm['total_score_weighted'] = (
                            rfm['R'].astype(int)* weights['R'] 
                            + rfm['F'].astype(int)*weights['F'] 
                            + rfm['M'].astype(int)*weights['M']
                            )
rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M,total_score,total_score_weighted
0,Abbey O'Reilly DVM,204,6,472,3,3,3,9,3.0
1,Add Senger,139,3,340,3,1,2,6,2.5
2,Aden Lesch Sr.,193,4,405,3,2,2,7,2.7
3,Admiral Senger,131,5,448,4,2,3,9,3.5
4,Agness O'Keefe,89,9,843,4,4,4,12,4.0


In [17]:
rfm['R'].dtype

CategoricalDtype(categories=['4', '3', '2', '1'], ordered=True, categories_dtype=str)

In [19]:
rfm['segment'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm.head()


,customer_id,Recency,Frequency,Monetary,R,F,M,total_score,total_score_weighted,segment
0,Abbey O'Reilly DVM,204,6,472,3,3,3,9,3.0,333
1,Add Senger,139,3,340,3,1,2,6,2.5,312
2,Aden Lesch Sr.,193,4,405,3,2,2,7,2.7,322
3,Admiral Senger,131,5,448,4,2,3,9,3.5,423
4,Agness O'Keefe,89,9,843,4,4,4,12,4.0,444


In [20]:
rfm["segment"].value_counts()

segment
111    90
444    62
344    58
322    44
222    44
211    38
422    37
223    36
244    34
311    31
112    31
122    29
323    28
411    28
123    27
433    26
423    25
333    20
421    18
443    18
212    17
321    17
133    16
234    15
144    15
233    14
221    13
334    13
121    11
343    11
224    11
232    11
312    10
434    10
124    10
332     9
424     8
432     7
134     6
113     6
412     6
324     5
213     5
243     5
143     4
313     3
413     3
442     2
142     2
342     1
431     1
241     1
131     1
242     1
132     1
Name: count, dtype: int64

In [21]:
rfm["segment"].value_counts(normalize=True)

segment
111    0.090452
444    0.062312
344    0.058291
322    0.044221
222    0.044221
211    0.038191
422    0.037186
223    0.036181
244    0.034171
311    0.031156
112    0.031156
122    0.029146
323    0.028141
411    0.028141
123    0.027136
433    0.026131
423    0.025126
333    0.020101
421    0.018090
443    0.018090
212    0.017085
321    0.017085
133    0.016080
234    0.015075
144    0.015075
233    0.014070
221    0.013065
334    0.013065
121    0.011055
343    0.011055
224    0.011055
232    0.011055
312    0.010050
434    0.010050
124    0.010050
332    0.009045
424    0.008040
432    0.007035
134    0.006030
113    0.006030
412    0.006030
324    0.005025
213    0.005025
243    0.005025
143    0.004020
313    0.003015
413    0.003015
442    0.002010
142    0.002010
342    0.001005
431    0.001005
241    0.001005
131    0.001005
242    0.001005
132    0.001005
Name: proportion, dtype: float64

In [32]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("Recency", "Frequency", "Monetary")
)

fig.add_trace(
    go.Histogram(x=rfm["Recency"], nbinsx=30, name=""),
    row=1,
    col=1
)

fig.add_trace(
    go.Histogram(x=rfm["Frequency"], nbinsx=30, name=""),
    row=1,
    col=2
)

fig.add_trace(
    go.Histogram(x=rfm["Monetary"], nbinsx=30, name=""),
    row=1,
    col=3
)

fig.update_layout(
    title="",
    showlegend=False
)

fig.update_xaxes(title_text="", row=1, col=1)
fig.update_xaxes(title_text="", row=1, col=2)
fig.update_xaxes(title_text="", row=1, col=3)

fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="", row=1, col=2)
fig.update_yaxes(title_text="", row=1, col=3)

fig.show()

In [36]:
def naming(df:pd.DataFrame,rfm_score:str):
    if df[rfm_score] >= 9:
        return 'Can\'t Loose Them'
    elif ((df[rfm_score] >= 8) and (df[rfm_score] < 9)):
        return 'Champions'
    elif ((df[rfm_score] >= 7) and (df[rfm_score] < 8)):
        return 'Loyal/Commited'
    elif ((df[rfm_score] >= 6) and (df[rfm_score] < 7)):
        return 'Potential'
    elif ((df[rfm_score] >= 5) and (df[rfm_score] < 6)):
        return 'Promising'
    elif ((df[rfm_score] >= 4) and (df[rfm_score] < 5)):
        return 'Requires Attention'
    else:
        return 'Demands Activation'

In [35]:
rfm.head()

,customer_id,Recency,Frequency,Monetary,R,F,M,total_score,total_score_weighted,segment
0,Abbey O'Reilly DVM,204,6,472,3,3,3,9,3.0,333
1,Add Senger,139,3,340,3,1,2,6,2.5,312
2,Aden Lesch Sr.,193,4,405,3,2,2,7,2.7,322
3,Admiral Senger,131,5,448,4,2,3,9,3.5,423
4,Agness O'Keefe,89,9,843,4,4,4,12,4.0,444


In [37]:
rfm['Segment_Name'] = rfm.apply(naming, args={'total_score'} ,axis=1)
rfm.head(5)

,customer_id,Recency,Frequency,Monetary,R,F,M,total_score,total_score_weighted,segment,Segment_Name
0,Abbey O'Reilly DVM,204,6,472,3,3,3,9,3.0,333,Can't Loose Them
1,Add Senger,139,3,340,3,1,2,6,2.5,312,Potential
2,Aden Lesch Sr.,193,4,405,3,2,2,7,2.7,322,Loyal/Commited
3,Admiral Senger,131,5,448,4,2,3,9,3.5,423,Can't Loose Them
4,Agness O'Keefe,89,9,843,4,4,4,12,4.0,444,Can't Loose Them


In [40]:
grouped_by = rfm.groupby('Segment_Name').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count'] # `count`- ը պետք էր ինչ-որ տեղ նշել , կապ չունի monetary-ի հետ
 }).round(1)

grouped_by.head()


grouped_by.columns = ['RecencyMean', 'FrequencyMean', 'MonetaryMean', 'Count']
grouped_by = grouped_by.reset_index()

grouped_by

,Segment_Name,RecencyMean,FrequencyMean,MonetaryMean,Count
0,Can't Loose Them,166.6,7.2,707.2,335
1,Champions,212.0,5.0,489.6,114
2,Demands Activation,600.5,1.9,151.0,90
3,Loyal/Commited,275.5,4.7,429.3,147
4,Potential,286.8,3.9,352.1,132
5,Promising,361.4,3.4,294.8,97
6,Requires Attention,460.7,2.8,246.0,80


In [42]:
fig = px.treemap(
    grouped_by,
    path=['Segment_Name'],
    values='Count', # որոշում էի. պատկերի չափսը
    color='RecencyMean',
    hover_data=['RecencyMean', 'FrequencyMean', 'MonetaryMean'],
    title='RFM Segments'
)

fig.show()